# A0 Baseline Pretrain - Colab Resume Notebook

A0 baseline 본 실험용 노트북입니다. smoke 설정을 쓰지 않고 전체 train/validation 데이터와 아래 `TRAIN_CONFIG`로 실행합니다.

- checkpoint, tokenizer, history는 Google Drive의 `gpt-lab-runs/A0_baseline`에 저장합니다.
- Colab 연결이 끊기면 같은 노트북을 다시 열고 위에서부터 실행한 뒤 학습 셀을 다시 실행하면 `latest_checkpoint.pt`에서 이어서 시작합니다.
- epoch 중간에 끊겨도 epoch별 shuffle seed를 고정하고 완료한 batch 수를 저장하므로 이미 끝난 batch를 건너뜁니다.

## 1. Runtime

Colab 메뉴에서 `Runtime > Change runtime type > GPU`를 선택한 뒤 실행하세요. 가능하면 GPU 종류도 결과표에 기록합니다.

In [2]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = 'https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git'

IN_COLAB = False
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    IN_COLAB = True
    try:
        drive.mount('/content/drive')
        DRIVE_MOUNTED = Path('/content/drive/MyDrive').exists()
    except Exception as exc:
        print('Google Drive mount skipped:', exc)
        print('Drive 저장이 필요하면 이 셀을 다시 실행하거나 Colab 웹에서 Drive 권한을 다시 승인하세요.')
except Exception as exc:
    print('google.colab not available:', exc)

project_candidates = [
    Path.cwd(),
    Path('/content/gpt-lab'),
    Path('/content/week14-team-05-gpt-lab'),
    Path('/content/drive/MyDrive/gpt-lab'),
    Path('/content/drive/MyDrive/week14-team-05-gpt-lab'),
]

PROJECT_DIR = next((path for path in project_candidates if (path / 'src').exists()), None)
if PROJECT_DIR is None and Path('/content').exists():
    PROJECT_DIR = Path('/content/gpt-lab')
    if not PROJECT_DIR.exists():
        print('Project repo not found. Cloning:', REPO_URL)
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

if PROJECT_DIR is None or not (PROJECT_DIR / 'src').exists():
    raise RuntimeError(
        'Project src directory not found. Clone the repo first or set PROJECT_DIR to the repo path.'
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR:', PROJECT_DIR)
print('DRIVE_MOUNTED:', DRIVE_MOUNTED)

if (PROJECT_DIR / 'requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


Mounted at /content/drive
Project repo not found. Cloning: https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git
PROJECT_DIR: /content/gpt-lab
DRIVE_MOUNTED: True


## 2. Config

In [3]:
BASE_CONFIG = {
    'vocab_size': 3000,
    'context_length': 64,
    'emb_dim': 128,
    'n_heads': 4,
    'n_layers': 2,
    'drop_rate': 0.1,
    'qkv_bias': False,
}

TRAIN_CONFIG = {
    'seed': 42,
    'batch_size': 8,
    'learning_rate': 3e-4,
    'weight_decay': 0.0,
    'num_epochs': 2,
    'eval_freq': 100,
    'eval_iter': 10,
    'start_context': '영화',
}

EXPERIMENT_ID = 'A0_baseline'
RUN_DATE = None  # None이면 오늘 날짜 YYYYMMDD를 사용합니다.

print('BASE_CONFIG:', BASE_CONFIG)
print('TRAIN_CONFIG:', TRAIN_CONFIG)


BASE_CONFIG: {'vocab_size': 3000, 'context_length': 64, 'emb_dim': 128, 'n_heads': 4, 'n_layers': 2, 'drop_rate': 0.1, 'qkv_bias': False}
TRAIN_CONFIG: {'seed': 42, 'batch_size': 8, 'learning_rate': 0.0003, 'weight_decay': 0.0, 'num_epochs': 2, 'eval_freq': 100, 'eval_iter': 10, 'start_context': '영화'}


## 3. Imports, Paths, Seed

In [4]:
import json
import math
import random
import time
from datetime import date

import numpy as np
import torch

import download_data
from src.bpe import BPETokenizer
from src.dataset import GPTDataset
from src.model import GPTModel
from src.train import calc_loss_batch, calc_loss_loader, generate_and_print_sample, save_checkpoint

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(TRAIN_CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'

if IN_COLAB and Path('/content/drive/MyDrive').exists():
    RUN_DIR = Path('/content/drive/MyDrive/gpt-lab-runs') / EXPERIMENT_ID
else:
    RUN_DIR = PROJECT_DIR / 'runs' / EXPERIMENT_ID

DATA_DIR = PROJECT_DIR / 'data'
CKPT_DIR = RUN_DIR / 'checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

run_date = RUN_DATE or date.today().strftime('%Y%m%d')
TOKENIZER_PATH = RUN_DIR / f"vocab_bpe_{BASE_CONFIG['vocab_size']}.json"
LATEST_CKPT_PATH = CKPT_DIR / 'latest_checkpoint.pt'
BEST_CKPT_PATH = CKPT_DIR / f'{EXPERIMENT_ID}_{run_date}_best.pt'
HISTORY_PATH = RUN_DIR / 'history.json'
CONFIG_PATH = RUN_DIR / 'config.json'

print('device:', device)
print('gpu:', gpu_name)
print('RUN_DIR:', RUN_DIR)
print('LATEST_CKPT_PATH:', LATEST_CKPT_PATH)
print('BEST_CKPT_PATH:', BEST_CKPT_PATH)


device: cuda
gpu: Tesla T4
RUN_DIR: /content/drive/MyDrive/gpt-lab-runs/A0_baseline
LATEST_CKPT_PATH: /content/drive/MyDrive/gpt-lab-runs/A0_baseline/checkpoints/latest_checkpoint.pt
BEST_CKPT_PATH: /content/drive/MyDrive/gpt-lab-runs/A0_baseline/checkpoints/A0_baseline_20260602_best.pt


## 4. Data and Tokenizer

전체 train/validation 텍스트를 사용합니다. smoke용 character limit은 없습니다.

In [5]:
train_path = DATA_DIR / 'nsmc_lm_train.txt'
val_path = DATA_DIR / 'nsmc_lm_val.txt'
if not train_path.exists() or not val_path.exists():
    print('LM data not found. Running download_data.main()...')
    download_data.main()

train_text = train_path.read_text(encoding='utf-8')
val_text = val_path.read_text(encoding='utf-8')
print('train chars:', len(train_text))
print('val chars:', len(val_text))

tokenizer = BPETokenizer(vocab_size=BASE_CONFIG['vocab_size'])
if TOKENIZER_PATH.exists():
    tokenizer.load(TOKENIZER_PATH)
    print('loaded tokenizer:', TOKENIZER_PATH)
else:
    tokenizer.train(train_text)
    tokenizer.save(TOKENIZER_PATH)
    print('saved tokenizer:', TOKENIZER_PATH)

train_ids = tokenizer.encode(train_text)
val_ids = tokenizer.encode(val_text)
print('train tokens:', len(train_ids))
print('val tokens:', len(val_ids))


LM data not found. Running download_data.main()...
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
저장됨: /content/gpt-lab/data/ratings_train.txt
다운로드 중: https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
저장됨: /content/gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /content/gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
train chars: 1379486
val chars: 120560
saved tokenizer: /content/drive/MyDrive/gpt-lab-runs/A0_baseline/vocab_bpe_3000.json
train tokens: 805021
val tokens: 70386


## 5. Resume-Aware Training Loop

이 셀을 다시 실행하면 `latest_checkpoint.pt`가 있을 때 자동으로 이어서 학습합니다.

In [ ]:
def finite_or_raise(name, value):
    if not math.isfinite(float(value)):
        raise RuntimeError(f'{name} is not finite: {value}')

def make_train_loader(epoch):
    generator = torch.Generator()
    generator.manual_seed(TRAIN_CONFIG['seed'] + epoch)
    dataset = GPTDataset(
        train_ids,
        context_length=BASE_CONFIG['context_length'],
        stride=BASE_CONFIG['context_length'],
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=TRAIN_CONFIG['batch_size'],
        shuffle=True,
        drop_last=True,
        num_workers=0,
        generator=generator,
    )

def make_val_loader():
    dataset = GPTDataset(
        val_ids,
        context_length=BASE_CONFIG['context_length'],
        stride=BASE_CONFIG['context_length'],
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=TRAIN_CONFIG['batch_size'],
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )

def write_history(history):
    HISTORY_PATH.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')

def save_latest(model, optimizer, state):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        **state,
    }
    torch.save(checkpoint, LATEST_CKPT_PATH)

def load_latest(model, optimizer):
    if not LATEST_CKPT_PATH.exists():
        return None
    checkpoint = torch.load(LATEST_CKPT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint

config_record = {
    'experiment_id': EXPERIMENT_ID,
    'run_date': run_date,
    'base_config': BASE_CONFIG,
    'train_config': TRAIN_CONFIG,
    'gpu': gpu_name,
    'project_dir': str(PROJECT_DIR),
    'run_dir': str(RUN_DIR),
}
CONFIG_PATH.write_text(json.dumps(config_record, ensure_ascii=False, indent=2), encoding='utf-8')

model = GPTModel(BASE_CONFIG).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG['learning_rate'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
)
val_loader = make_val_loader()

history = {
    'train_losses': [],
    'val_losses': [],
    'eval_steps': [],
    'eval_train_losses': [],
    'eval_val_losses': [],
    'best_val_loss': float('inf'),
    'best_checkpoint_path': str(BEST_CKPT_PATH),
}

resume = load_latest(model, optimizer)
if resume is None:
    start_epoch = 0
    start_batch = 0
    global_step = 0
    best_val_loss = float('inf')
    epoch_loss_resume = 0.0
    batches_seen_resume = 0
    initial_val_loss = calc_loss_loader(val_loader, model, device, num_batches=TRAIN_CONFIG['eval_iter'])
    finite_or_raise('initial_val_loss', initial_val_loss)
    print(f'initial val loss: {initial_val_loss:.4f}')
else:
    start_epoch = int(resume['epoch'])
    start_batch = int(resume.get('batch_in_epoch', 0))
    global_step = int(resume['global_step'])
    best_val_loss = float(resume.get('best_val_loss', float('inf')))
    epoch_loss_resume = float(resume.get('epoch_loss', 0.0))
    batches_seen_resume = int(resume.get('batches_seen', 0))
    history.update(resume.get('history', {}))
    print(f"resuming from epoch={start_epoch}, batch={start_batch}, global_step={global_step}")
    print(f'best val loss so far: {best_val_loss:.4f}')

started_at = time.time()
try:
    for epoch in range(start_epoch, TRAIN_CONFIG['num_epochs']):
        train_loader = make_train_loader(epoch)
        model.train()

        if epoch == start_epoch and start_batch > 0:
            epoch_loss = epoch_loss_resume
            batches_seen = batches_seen_resume
            print(f'epoch {epoch + 1}: skipping first {start_batch} already-finished batches')
        else:
            start_batch = 0
            epoch_loss = 0.0
            batches_seen = 0

        for batch_idx, (input_batch, target_batch) in enumerate(train_loader):
            if batch_idx < start_batch:
                continue

            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            finite_or_raise('train_loss', loss.item())
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batches_seen += 1
            global_step += 1
            next_batch = batch_idx + 1

            if TRAIN_CONFIG['eval_freq'] > 0 and global_step % TRAIN_CONFIG['eval_freq'] == 0:
                train_eval_loss = calc_loss_loader(train_loader, model, device, TRAIN_CONFIG['eval_iter'])
                val_eval_loss = calc_loss_loader(val_loader, model, device, TRAIN_CONFIG['eval_iter'])
                finite_or_raise('train_eval_loss', train_eval_loss)
                finite_or_raise('val_eval_loss', val_eval_loss)
                history['eval_steps'].append(global_step)
                history['eval_train_losses'].append(train_eval_loss)
                history['eval_val_losses'].append(val_eval_loss)
                write_history(history)
                print(f'step {global_step}: train loss {train_eval_loss:.4f}, val loss {val_eval_loss:.4f}')

                save_latest(model, optimizer, {
                    'epoch': epoch,
                    'batch_in_epoch': next_batch,
                    'global_step': global_step,
                    'best_val_loss': best_val_loss,
                    'epoch_loss': epoch_loss,
                    'batches_seen': batches_seen,
                    'history': history,
                    'run_date': run_date,
                })

        avg_epoch_loss = epoch_loss / batches_seen if batches_seen else float('nan')
        val_loss = calc_loss_loader(val_loader, model, device)
        finite_or_raise('avg_epoch_loss', avg_epoch_loss)
        finite_or_raise('val_loss', val_loss)

        history['train_losses'].append(avg_epoch_loss)
        history['val_losses'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            history['best_val_loss'] = best_val_loss
            save_checkpoint(model, optimizer, epoch=epoch + 1, global_step=global_step, path=str(BEST_CKPT_PATH))

        write_history(history)
        save_latest(model, optimizer, {
            'epoch': epoch + 1,
            'batch_in_epoch': 0,
            'global_step': global_step,
            'best_val_loss': best_val_loss,
            'epoch_loss': 0.0,
            'batches_seen': 0,
            'history': history,
            'run_date': run_date,
        })

        print(f'epoch {epoch + 1}: train loss {avg_epoch_loss:.4f}, val loss {val_loss:.4f}, best val loss {best_val_loss:.4f}')
        generate_and_print_sample(
            model,
            tokenizer,
            device,
            TRAIN_CONFIG['start_context'],
            context_size=BASE_CONFIG['context_length'],
        )

except torch.cuda.OutOfMemoryError as exc:
    print('CUDA OOM. Try lowering batch_size only for a separate OOM recovery run; do not mix it into A0 records.')
    raise exc

elapsed = time.time() - started_at
print('DONE')
print('elapsed sec:', round(elapsed, 1))
print('best val loss:', history.get('best_val_loss'))
print('best checkpoint:', history.get('best_checkpoint_path'))
print('latest checkpoint:', LATEST_CKPT_PATH)


## 6. Results Summary

학습이 끝난 뒤 이 셀을 실행해 기록할 값을 확인합니다.

In [ ]:
if HISTORY_PATH.exists():
    history = json.loads(HISTORY_PATH.read_text(encoding='utf-8'))
    print(json.dumps({
        'experiment_id': EXPERIMENT_ID,
        'gpu': gpu_name,
        'train_losses': history.get('train_losses'),
        'val_losses': history.get('val_losses'),
        'best_val_loss': history.get('best_val_loss'),
        'best_checkpoint_path': history.get('best_checkpoint_path'),
        'latest_checkpoint_path': str(LATEST_CKPT_PATH),
    }, ensure_ascii=False, indent=2))
else:
    print('history.json not found yet:', HISTORY_PATH)
